In [1]:
import re
import nltk
import pandas as pd
from datasets import concatenate_datasets, load_dataset
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengunduh resource tokenizer dan stopword bahasa Indonesia dari NLTK
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

d:\TUGAS\TUGAS ORANG\Asril\tugas-nlp-preprocessing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to C:\Users\ASUS TUF
[nltk_data]     GAMING\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\ASUS TUF
[nltk_data]     GAMING\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\ASUS TUF
[nltk_data]     GAMING\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\ASUS TUF
[nltk_data]     GAMING\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [2]:
# 1. Memuat dataset dari Hugging Face
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")

# Menggabungkan split dataset (jika terbagi menjadi train/test/validation) ke dalam satu DataFrame
if len(dataset.keys()) > 1:
  df = concatenate_datasets(
      [dataset[split] for split in dataset.keys()]
  ).to_pandas()
else:
  split_awal = list(dataset.keys())[0]
  df = dataset[split_awal].to_pandas()

# 2. Menampilkan informasi dataset
print("=== INFORMASI DATASET ===")
print(f"Jumlah Total Data : {len(df)} baris")
print(f"Nama Kolom        : {df.columns.tolist()}")
print("\nDetail Tipe Data & Missing Values:")
print(df.info())

# Deteksi otomatis nama kolom teks dan label sentimen
kolom_teks = "review" if "review" in df.columns else df.columns[0]
kolom_label = "sentiment" if "sentiment" in df.columns else df.columns[-1]

print(f"\n=== DISTRIBUSI LABEL SENTIMEN ({kolom_label}) ===")
distribusi = df[kolom_label].value_counts()
persentase = df[kolom_label].value_counts(normalize=True) * 100
ringkasan_label = pd.DataFrame(
    {"Jumlah Data": distribusi, "Persentase (%)": persentase.round(2)}
)
display(ringkasan_label)

=== INFORMASI DATASET ===
Jumlah Total Data : 11606 baris
Nama Kolom        : ['review', 'translate', 'sentimen']

Detail Tipe Data & Missing Values:
<class 'pandas.DataFrame'>
RangeIndex: 11606 entries, 0 to 11605
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     11606 non-null  str  
 1   translate  11606 non-null  str  
 2   sentimen   11606 non-null  int64
dtypes: int64(1), str(2)
memory usage: 2.6 MB
None

=== DISTRIBUSI LABEL SENTIMEN (sentimen) ===


,Jumlah Data,Persentase (%)
sentimen,,
0,5803,50.0
1,5803,50.0


In [3]:
# Menyiapkan daftar stopword gabungan (NLTK + Sastrawi)
stopword_nltk = set(stopwords.words("indonesian"))
stopword_sastrawi = set(StopWordRemoverFactory().get_stop_words())
daftar_stopword = stopword_nltk.union(stopword_sastrawi)

# Menyiapkan stemmer bahasa Indonesia (Sastrawi)
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stem_cache = {}  # Cache agar kata yang sama tidak di-stem berulang kali


def stem_kata(kata):
  if kata not in stem_cache:
    stem_cache[kata] = stemmer.stem(kata)
  return stem_cache[kata]


# Fungsi tahapan preprocessing
def case_folding(teks):
  return str(teks).lower()


def cleaning(teks):
  teks = re.sub(r"http\S+|www\S+|https\S+", "", teks)  # Hapus URL
  teks = re.sub(r"\d+", "", teks)  # Hapus angka
  teks = re.sub(
      r"[^a-z\s]", " ", teks
  )  # Hapus tanda baca, emoji, & karakter khusus
  teks = re.sub(r"\s+", " ", teks).strip()  # Rapikan spasi ganda
  return teks


def tokenisasi(teks):
  return word_tokenize(teks)


def stopword_removal(tokens):
  return [kata for kata in tokens if kata not in daftar_stopword]


def stemming(tokens):
  return [stem_kata(kata) for kata in tokens]


# Menjalankan seluruh rangkaian preprocessing secara bertahap
df["1_case_folding"] = df[kolom_teks].apply(case_folding)
df["2_cleaning"] = df["1_case_folding"].apply(cleaning)
df["3_tokenisasi"] = df["2_cleaning"].apply(tokenisasi)
df["4_stopword_removal"] = df["3_tokenisasi"].apply(stopword_removal)
df["5_stemming_tokens"] = df["4_stopword_removal"].apply(stemming)

# Menggabungkan kembali hasil akhir token menjadi kalimat bersih
df["teks_bersih"] = df["5_stemming_tokens"].apply(lambda tokens: " ".join(tokens))

print("Proses preprocessing selesai!")

Proses preprocessing selesai!


In [4]:
# Mengatur agar seluruh isi teks tampil penuh tanpa terpotong
pd.set_option("display.max_colwidth", None)

hasil_perbandingan = df[[kolom_teks, "teks_bersih"]].head(10).copy()
hasil_perbandingan.columns = [
    "Teks Sebelum Preprocessing",
    "Teks Sesudah Preprocessing",
]

# Menampilkan 10 data pertama dalam tabel
display(hasil_perbandingan)

,Teks Sebelum Preprocessing,Teks Sesudah Preprocessing
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg ga nyala
1,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek tolong perhati
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk harga jangkau kirim cepet banget udh order oke tp sayang klo kado gabisa nitip pake kartu ucap bungkus pake kertas kado
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol fungsi packaging penyot robek
5,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk original kualitas cepat kirim sanagt cepat recommended seller
6,kualitas sangat jelek. bocor,kualitas jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih ka barang terima sukses ka
8,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller",body berat cm spiral tipis banding cm moga awet beli cm penasaran aja cm udah rusak mesin harga murah cm fast delivery courier ramah ty seller
9,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesan terima tanggal january buka pagi tanggal january tab sketsa advan nyala cas posisi tab nyala
